# Sliding Window Attention (滑动窗口注意力) 实现

滑动窗口注意力限制每个token只关注固定窗口内的邻近token。

**核心思想**：
- 限制注意力范围到固定大小的窗口
- 降低计算复杂度从O(n²)到O(n×w)
- 适合处理长序列

**公式**：
$$
\text{Attention}_i = \text{softmax}\left(\frac{Q_i \cdot K_{[i-w:i+w]}}{\sqrt{d_k}}\right) \cdot V_{[i-w:i+w]}
$$
其中$w$是窗口大小。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')

## 1. 辅助函数

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def create_sliding_window_mask(seq_len, window_size):
    """
    创建滑动窗口mask
    
    Args:
        seq_len: 序列长度
        window_size: 窗口大小（每侧的范围）
    
    Returns:
        mask: 滑动窗口mask (seq_len, seq_len)
    """
    mask = np.zeros((seq_len, seq_len))
    
    for i in range(seq_len):
        start = max(0, i - window_size)
        end = min(seq_len, i + window_size + 1)
        mask[i, start:end] = 1
    
    return mask


def create_sliding_window_mask_one_sided(seq_len, window_size):
    """创建单侧滑动窗口mask（只看左侧）"""
    mask = np.zeros((seq_len, seq_len))
    
    for i in range(seq_len):
        start = max(0, i - window_size)
        end = i + 1
        mask[i, start:end] = 1
    
    return mask

## 2. 可视化滑动窗口Mask

In [ ]:
# 生成不同窗口大小的mask
seq_len = 16
window_sizes = [1, 2, 4]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, ws in enumerate(window_sizes):
    mask = create_sliding_window_mask(seq_len, ws)
    
    sns.heatmap(mask, annot=False, cmap='Blues', cbar=False,
                ax=axes[idx], square=True, vmin=0, vmax=1)
    axes[idx].set_title(f'窗口大小={ws} (总窗口={2*ws+1})', 
                        fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Key位置')
    axes[idx].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("观察:")
print("  • 蓝色区域表示可见（mask=1）")
print("  • 白色区域表示被mask（mask=0）")
print("  • 窗口越大，可见范围越广")
print("  • 注意力集中在对角线附近（局部建模）")

## 3. 实现Sliding Window Attention类

In [ ]:
class SlidingWindowAttention:
    """
    Sliding Window Attention (滑动窗口注意力)
    
    每个token只关注固定窗口内的token
    """
    
    def __init__(self, embed_dim, num_heads, window_size, use_bias=True):
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.window_size = window_size
        self.use_bias = use_bias
        
        # Q, K, V投影
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(embed_dim)
            self.b_v = np.zeros(embed_dim)
            self.b_o = np.zeros(embed_dim)
    
    def split_heads(self, x):
        """分割成多个头"""
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 0, 2)
        return x
    
    def combine_heads(self, x):
        """合并多个头"""
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.embed_dim)
        return x
    
    def sliding_window_attention(self, Q, K, V, window_mask=None):
        """计算滑动窗口注意力"""
        num_heads, seq_len, head_dim = Q.shape
        
        # 1. 计算注意力得分
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(head_dim)
        
        # 2. 应用滑动窗口mask
        if window_mask is None:
            window_mask = create_sliding_window_mask(seq_len, self.window_size)
        
        window_mask = window_mask[None, :, :]
        scores = np.where(window_mask == 0, -1e9, scores)
        
        # 3. Softmax
        attention_weights = softmax(scores, axis=-1)
        
        # 4. 加权求和
        output = np.matmul(attention_weights, V)
        
        return output, attention_weights
    
    def forward(self, query, key=None, value=None, window_mask=None, return_attention=False):
        """前向传播"""
        if key is None:
            key = query
        if value is None:
            value = key
        
        # 1. 线性投影
        Q = np.dot(query, self.W_q)
        K = np.dot(key, self.W_k)
        V = np.dot(value, self.W_v)
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # 2. 分割头
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 3. 滑动窗口注意力
        multi_head_output, attention_weights = self.sliding_window_attention(
            Q, K, V, window_mask=window_mask
        )
        
        # 4. 合并头
        concatenated = self.combine_heads(multi_head_output)
        
        # 5. 输出投影
        output = np.dot(concatenated, self.W_o)
        if self.use_bias:
            output += self.b_o
        
        if return_attention:
            return output, attention_weights
        
        return output


class SlidingWindowSelfAttention(SlidingWindowAttention):
    """滑动窗口自注意力（简化接口）"""
    
    def forward(self, x, window_mask=None, return_attention=False):
        return super().forward(x, x, x, window_mask=window_mask, 
                              return_attention=return_attention)

## 4. 测试滑动窗口注意力

In [ ]:
# 参数设置
seq_len = 16
embed_dim = 64
num_heads = 4
window_size = 2

print("配置:")
print(f"  序列长度: {seq_len}")
print(f"  嵌入维度: {embed_dim}")
print(f"  注意力头数: {num_heads}")
print(f"  窗口大小: {window_size} (每侧)")
print(f"  总窗口大小: {2*window_size+1}")

# 生成输入
x = np.random.randn(seq_len, embed_dim)

# 创建滑动窗口注意力层
swa = SlidingWindowSelfAttention(embed_dim, num_heads, window_size)

# 前向传播
output, attn_weights = swa.forward(x, return_attention=True)

print(f"\n形状:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  注意力权重: {attn_weights.shape}")

## 5. 可视化注意力模式（稀疏性）

In [ ]:
# 可视化所有头的注意力
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i in range(4):
    sns.heatmap(attn_weights[i], annot=False, cmap='YlOrRd',
                ax=axes[i], cbar=True, square=True, vmin=0, vmax=0.4)
    axes[i].set_title(f'注意力头 {i+1} (窗口大小={window_size})', 
                      fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Key位置')
    axes[i].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

# 统计稀疏度
total_elements = attn_weights[0].size
nonzero_elements = np.count_nonzero(attn_weights[0] > 1e-6)
sparsity = (1 - nonzero_elements / total_elements) * 100

print("\n稀疏度分析:")
print(f"  总元素数: {total_elements}")
print(f"  非零元素数: {nonzero_elements}")
print(f"  稀疏度: {sparsity:.1f}%")
print("\n观察:")
print("  • 注意力集中在对角线附近（带状模式）")
print("  • 窗口外的注意力权重为0（稀疏）")
print("  • 稀疏性带来计算和内存优势")

## 6. 对比不同窗口大小

In [ ]:
# 创建不同窗口大小的模型
window_sizes_test = [1, 2, 4, 8]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, ws in enumerate(window_sizes_test):
    swa_test = SlidingWindowSelfAttention(embed_dim, num_heads, ws)
    _, attn_test = swa_test.forward(x, return_attention=True)
    
    sns.heatmap(attn_test[0], annot=False, cmap='YlOrRd',
                ax=axes[idx], cbar=True, square=True, vmin=0, vmax=0.5)
    axes[idx].set_title(f'窗口大小={ws} (总窗口={2*ws+1})', 
                        fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Key位置')
    axes[idx].set_ylabel('Query位置')
    
    # 计算稀疏度
    sparsity = (1 - np.count_nonzero(attn_test[0] > 1e-6) / attn_test[0].size) * 100
    axes[idx].text(0.02, 0.98, f'稀疏度: {sparsity:.1f}%', 
                   transform=axes[idx].transAxes, fontsize=10,
                   verticalalignment='top', 
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("窗口大小的影响:")
print(f"\n{'窗口大小':>10} {'总窗口':>10} {'稀疏度':>10}")
print("-" * 35)
for ws in window_sizes_test:
    total_window = 2 * ws + 1
    mask = create_sliding_window_mask(seq_len, ws)
    avg_visible = mask.sum() / seq_len
    sparsity_pct = (1 - avg_visible / seq_len) * 100
    print(f"{ws:>10} {total_window:>10} {sparsity_pct:>9.1f}%")

print("\n权衡:")
print("  • 窗口越小: 计算越快，但上下文越少")
print("  • 窗口越大: 上下文越丰富，但计算越慢")

## 7. 单侧窗口（自回归生成）

In [ ]:
# 创建单侧窗口mask
one_sided_mask = create_sliding_window_mask_one_sided(seq_len, window_size)

# 使用单侧窗口
output_one_sided, attn_one_sided = swa.forward(
    x, window_mask=one_sided_mask, return_attention=True
)

# 可视化对比
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# 双向窗口mask
bidirectional_mask = create_sliding_window_mask(seq_len, window_size)
sns.heatmap(bidirectional_mask, annot=False, cmap='Blues',
            ax=ax1, cbar=False, square=True)
ax1.set_title('双向窗口Mask', fontsize=12, fontweight='bold')
ax1.set_xlabel('Key位置')
ax1.set_ylabel('Query位置')

# 单侧窗口mask
sns.heatmap(one_sided_mask, annot=False, cmap='Greens',
            ax=ax2, cbar=False, square=True)
ax2.set_title('单侧窗口Mask（只看左侧）', fontsize=12, fontweight='bold')
ax2.set_xlabel('Key位置')
ax2.set_ylabel('Query位置')

# 单侧窗口注意力
sns.heatmap(attn_one_sided[0], annot=False, cmap='YlOrRd',
            ax=ax3, cbar=True, square=True)
ax3.set_title('单侧窗口注意力权重', fontsize=12, fontweight='bold')
ax3.set_xlabel('Key位置')
ax3.set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("单侧窗口的应用:")
print("  • 自回归语言模型（GPT风格）")
print("  • 限制只能看到历史信息")
print("  • 结合滑动窗口和因果约束")
print(f"  • 窗口大小{window_size}: 只看前{window_size}个token")

## 8. 计算复杂度分析

In [ ]:
# 对比不同序列长度下的计算量
seq_lengths = [128, 512, 1024, 4096, 16384]
window_size_compare = 128

standard_ops = [n * n for n in seq_lengths]
sliding_ops = [n * (2 * window_size_compare + 1) for n in seq_lengths]

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：绝对计算量
x_pos = np.arange(len(seq_lengths))
width = 0.35

bars1 = ax1.bar(x_pos - width/2, standard_ops, width, 
                label='标准注意力', alpha=0.7, color='red')
bars2 = ax1.bar(x_pos + width/2, sliding_ops, width,
                label='滑动窗口', alpha=0.7, color='green')

ax1.set_xlabel('序列长度', fontsize=12)
ax1.set_ylabel('操作数', fontsize=12)
ax1.set_title('计算量对比', fontsize=13, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(seq_lengths)
ax1.legend()
ax1.set_yscale('log')
ax1.grid(axis='y', alpha=0.3)

# 右图：加速比
speedups = [standard / sliding for standard, sliding in zip(standard_ops, sliding_ops)]
ax2.plot(seq_lengths, speedups, marker='o', linewidth=2, 
         markersize=10, color='blue')
ax2.set_xlabel('序列长度', fontsize=12)
ax2.set_ylabel('加速比', fontsize=12)
ax2.set_title('加速比 vs 序列长度', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 添加数值标签
for i, (length, speedup) in enumerate(zip(seq_lengths, speedups)):
    ax2.text(length, speedup, f'{speedup:.1f}x', 
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# 打印详细数据
print(f"计算复杂度对比 (窗口大小={window_size_compare}):\n")
print(f"{'序列长度':>10} {'标准注意力':>15} {'滑动窗口':>15} {'加速比':>10} {'内存节省':>10}")
print("-" * 65)

for i, n in enumerate(seq_lengths):
    speedup = speedups[i]
    memory_saving = (1 - sliding_ops[i] / standard_ops[i]) * 100
    print(f"{n:>10} {standard_ops[i]:>15,} {sliding_ops[i]:>15,} "
          f"{speedup:>9.1f}x {memory_saving:>9.1f}%")

print("\n关键观察:")
print("  ✓ 序列越长，滑动窗口的优势越明显")
print("  ✓ 复杂度从O(n²)降到O(n×w)")
print("  ✓ 16K序列时加速64倍以上")

## 9. 实际应用：Mistral 7B配置

In [ ]:
# Mistral 7B使用滑动窗口注意力
mistral_embed_dim = 4096
mistral_num_heads = 32
mistral_window_size = 4096
mistral_seq_len = 32768

print("Mistral 7B配置:\n")
print(f"  嵌入维度: {mistral_embed_dim}")
print(f"  注意力头数: {mistral_num_heads}")
print(f"  窗口大小: {mistral_window_size}")
print(f"  支持序列长度: {mistral_seq_len}")

# 计算复杂度
mistral_standard_ops = mistral_seq_len * mistral_seq_len
mistral_sliding_ops = mistral_seq_len * (2 * mistral_window_size + 1)
mistral_speedup = mistral_standard_ops / mistral_sliding_ops

print(f"\n复杂度对比:")
print(f"  标准注意力: {mistral_standard_ops:,} 操作")
print(f"  滑动窗口: {mistral_sliding_ops:,} 操作")
print(f"  加速比: {mistral_speedup:.1f}x")
print(f"  内存节省: {(1 - mistral_sliding_ops/mistral_standard_ops)*100:.1f}%")

print(f"\nMistral的设计选择:")
print(f"  ✓ 窗口{mistral_window_size}提供充足的局部上下文")
print(f"  ✓ 可处理{mistral_seq_len}长度的序列")
print(f"  ✓ 比标准注意力快{mistral_speedup:.0f}倍")
print(f"  ✓ 适合长文档和对话场景")

# 可视化窗口覆盖范围
positions = np.arange(0, mistral_seq_len, mistral_seq_len // 10)
plt.figure(figsize=(14, 3))

for pos in positions:
    start = max(0, pos - mistral_window_size)
    end = min(mistral_seq_len, pos + mistral_window_size + 1)
    plt.plot([start, end], [pos, pos], linewidth=3, alpha=0.6)
    plt.scatter(pos, pos, s=100, color='red', zorder=5)

plt.xlabel('Token位置', fontsize=12)
plt.ylabel('Query位置', fontsize=12)
plt.title('Mistral滑动窗口覆盖范围示意', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n图示说明:")
print("  • 红点: Query位置")
print("  • 横线: 该Query可见的窗口范围")
print(f"  • 每条线长度约{2*mistral_window_size}个token")

## 10. 混合策略：全局token + 滑动窗口

In [ ]:
# Longformer风格：部分token使用全局注意力
test_len = 12
ws = 1

# 创建混合mask
mixed_mask = create_sliding_window_mask(test_len, ws)

# 让第0个位置（如[CLS]）使用全局注意力
mixed_mask[0, :] = 1  # [CLS]可以看到所有位置
mixed_mask[:, 0] = 1  # 所有位置都能看到[CLS]

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 纯滑动窗口
pure_window = create_sliding_window_mask(test_len, ws)
sns.heatmap(pure_window, annot=True, fmt='.0f', cmap='Blues',
            ax=ax1, cbar=False, square=True)
ax1.set_title('纯滑动窗口', fontsize=12, fontweight='bold')
ax1.set_xlabel('Key位置')
ax1.set_ylabel('Query位置')

# 混合策略
sns.heatmap(mixed_mask, annot=True, fmt='.0f', cmap='Greens',
            ax=ax2, cbar=False, square=True)
ax2.set_title('混合策略（位置0为全局）', fontsize=12, fontweight='bold')
ax2.set_xlabel('Key位置')
ax2.set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("Longformer混合策略:")
print("  • 大部分token: 滑动窗口（局部注意力）")
print("  • 特殊token: 全局注意力（如[CLS], [SEP]）")
print("  • 任务相关token: 可选全局注意力")
print("\n优势:")
print("  ✓ 保持局部建模的效率")
print("  ✓ 通过全局token传递全局信息")
print("  ✓ 平衡效率和效果")

# 使用混合mask进行注意力计算
x_test = np.random.randn(test_len, embed_dim)
swa_test = SlidingWindowSelfAttention(embed_dim, num_heads, ws)
output_mixed, attn_mixed = swa_test.forward(
    x_test, window_mask=mixed_mask, return_attention=True
)

plt.figure(figsize=(8, 6))
sns.heatmap(attn_mixed[0], annot=True, fmt='.2f', cmap='YlOrRd',
            cbar=True, square=True)
plt.title('混合策略注意力权重', fontsize=13, fontweight='bold')
plt.xlabel('Key位置')
plt.ylabel('Query位置')
plt.tight_layout()
plt.show()

print("\n观察:")
print("  • 第0行: [CLS]对所有位置的注意力")
print("  • 第0列: 所有位置对[CLS]的注意力")
print("  • 其他位置: 局部滑动窗口注意力")

## 11. 滑动窗口 vs 标准注意力对比

In [ ]:
# 创建对比数据
import pandas as pd

comparison_data = {
    '特性': ['计算复杂度', '内存占用', '长序列能力', '局部建模', '全局建模', '稀疏性'],
    '标准注意力': ['O(n²)', '高', '受限', '良好', '优秀', '密集'],
    '滑动窗口': ['O(n×w)', '低', '优秀', '优秀', '有限', '稀疏']
}

df = pd.DataFrame(comparison_data)

print("滑动窗口 vs 标准注意力对比:\n")
print(df.to_string(index=False))

print("\n\n滑动窗口注意力的优势:")
print("  ✓ 线性复杂度: O(n×w)而非O(n²)")
print("  ✓ 内存高效: 只存储窗口内的注意力")
print("  ✓ 可扩展: 可处理超长序列")
print("  ✓ 局部建模: 保留邻近依赖关系")

print("\n局限性:")
print("  • 全局信息传递受限")
print("  • 需要多层堆叠扩大感受野")
print("  • 可能需要混合策略（如Longformer）")

print("\n应用场景:")
print("  • 长文档处理（Longformer）")
print("  • 长对话系统（Mistral）")
print("  • 音频/视频序列建模")
print("  • 任何需要处理长序列的场景")

## 总结

### 滑动窗口注意力的核心要点：

1. **核心机制**：
   - 限制注意力范围到固定大小的窗口
   - 每个token只关注邻近的w个token
   - 通过mask实现窗口限制

2. **关键优势**：
   - 线性复杂度O(n×w)
   - 内存占用显著降低
   - 可处理超长序列
   - 保留局部上下文

3. **实际应用**：
   - Mistral 7B: 窗口4096，序列32K
   - Longformer: 混合全局+局部
   - BigBird: 组合多种注意力模式

4. **设计选择**：
   - 窗口大小平衡效率和效果
   - 可选混合策略增强全局建模
   - 单侧窗口用于自回归生成

### 复杂度公式：
$$
\text{计算量} = n \times (2w + 1) = O(n \times w)
$$

### 加速比：
$$
\text{加速比} = \frac{n^2}{n \times w} = \frac{n}{w}
$$

序列越长，加速比越大！